# Monthly Liquidity Flow

This notebook presents the monthly cash flow calculated by the domain service. Credits are inflows, debits are outflows, and transactions without an operation value are excluded from monetary totals.

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(project_root))

from config import Config
from src.core.repositories import TransactionRepository
from src.core.services import calculate_monthly_cash_flow

In [ ]:
import pandas as pd
from IPython.display import display

config = Config()
transactions = TransactionRepository(config=config).all()
monthly_flow = calculate_monthly_cash_flow(transactions)

cash_flow_table = pd.DataFrame([
    {
        'month': flow.month,
        'inflow': flow.inflow,
        'outflow': flow.outflow,
        'net': flow.net,
        'transaction_count': flow.transaction_count,
    }
    for flow in monthly_flow
]).sort_values(['month']).reset_index(drop=True)

display(cash_flow_table)

In [ ]:
import matplotlib.pyplot as plt

months = [row.month.strftime('%Y-%m') for row in monthly_flow]
inflows = [float(row.inflow) for row in monthly_flow]
outflows = [float(row.outflow) for row in monthly_flow]
net = [float(row.net) for row in monthly_flow]

fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(months, inflows, label='Inflow', color='#2a9d8f')
ax.bar(months, [-value for value in outflows], label='Outflow', color='#e76f51')
ax.plot(months, net, label='Net', color='#264653', marker='o', linewidth=2)
ax.axhline(0, color='#333333', linewidth=0.8)
ax.set_title('Monthly Liquidity Flow')
ax.set_xlabel('Month')
ax.set_ylabel('Amount')
ax.legend()
ax.grid(axis='y', alpha=0.25)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()